# HealthMax — BanglaBERT NER Fine-tuning

**Run this notebook on Google Colab with T4 GPU.**

**Goal:** Fine-tune `sagorsarker/bangla-bert-base` on the BanglaHealthNER + MedER datasets for medical NER.

**BIO Labels:** `B-SYMPTOM`, `I-SYMPTOM`, `B-DISEASE`, `I-DISEASE`, `B-MEDICINE`, `I-MEDICINE`, `O`

**After training:** Upload checkpoint to S3 and update `NER_MODEL_PATH` in `.env`.

---

## Instructions for Collaborator
1. Click `Runtime > Change runtime type > GPU (T4)`.
2. Run cells from top to bottom.
3. Mount Google Drive if you want to save checkpoints there.
4. At the end, download `ner_model/` or upload to S3.

In [ ]:
# Cell 1: Install dependencies
%pip install -q transformers datasets seqeval accelerate
%pip install -q boto3 awscli  # For S3 upload at the end

In [ ]:
# Cell 2: Verify GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# Cell 3: Constants

MODEL_NAME = 'sagorsarker/bangla-bert-base'

# BIO label set — must match backend/ner.py LABEL_MAP exactly
LABEL_LIST = ['O', 'B-SYMPTOM', 'I-SYMPTOM', 'B-DISEASE', 'I-DISEASE', 'B-MEDICINE', 'I-MEDICINE']
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
OUTPUT_DIR = './ner_model'

print('Labels:', LABEL_LIST)

In [ ]:
# Cell 4: Load and convert NER dataset
# TODO (collaborator):
#   1. Upload BanglaHealthNER + MedER CSV files to Colab (or mount Drive).
#   2. Load CSV with pandas.
#   3. Convert to HuggingFace Dataset with columns: 'tokens' (list) and 'ner_tags' (list of int).
#   4. Split into train/validation (90/10).

# Placeholder — replace with real data loading
from datasets import Dataset, DatasetDict

# Example of expected format:
# tokens:   ['তিন', 'দিন', 'ধরে', 'জ্বর', ',', 'মাথাব্যথা']
# ner_tags: [0,     0,     0,     1,       0,  1           ]
#            O      O      O      B-SYMPTOM O   B-SYMPTOM

# TODO: Replace this dummy data with real dataset loading
dummy_data = {
    'tokens':   [['জ্বর', 'আছে'], ['বুকে', 'ব্যথা']],
    'ner_tags': [[1, 0],            [1, 0]           ],
}
raw_dataset = DatasetDict({
    'train':      Dataset.from_dict(dummy_data),
    'validation': Dataset.from_dict(dummy_data),
})
print('Dataset:', raw_dataset)

In [ ]:
# Cell 5: Load tokenizer and tokenize dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    """
    Tokenize words and align BIO labels to sub-word tokens.
    Sub-word tokens that continue a word get label -100 (ignored in loss).
    """
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
    )
    all_labels = []
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev_word_id:
                aligned.append(labels[word_id])
            else:
                aligned.append(-100)  # Ignore sub-word continuations
            prev_word_id = word_id
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized

tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=True)
print('Tokenized OK')

In [ ]:
# Cell 6: Load model and set up training
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import f1_score, classification_report

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    """Compute seqeval F1 for NER evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [LABEL_LIST[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [LABEL_LIST[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    return {'f1': f1_score(true_labels, true_preds)}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer ready. Starting training...')

In [ ]:
# Cell 7: Train the model
# TODO: Make sure real data is loaded in Cell 5 before running this.
train_result = trainer.train()
print('Training complete.', train_result)

In [ ]:
# Cell 8: Evaluate and print classification report
eval_results = trainer.evaluate()
print('Validation F1:', eval_results.get('eval_f1', 'N/A'))

In [ ]:
# Cell 9: Save the fine-tuned model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

# List saved files
import os
for f in os.listdir(OUTPUT_DIR):
    print(f)

In [ ]:
# Cell 10: Upload fine-tuned model to S3
# TODO (collaborator): Set your S3 bucket name and AWS credentials.

S3_BUCKET = 'healthmax-datasets'
S3_PREFIX = 'models/ner_model/'

# Uncomment to upload:
# !aws s3 sync {OUTPUT_DIR}/ s3://{S3_BUCKET}/{S3_PREFIX}
# print(f'Uploaded to s3://{S3_BUCKET}/{S3_PREFIX}')

print('Update NER_MODEL_PATH in .env to:', OUTPUT_DIR)
print('Or set it to the S3-synced local path on EC2.')